# Manuscript table review and workbook

This notebook previews the manuscript tables prepared by `main.py` and assembles the workbook. It runs independently after the main pipeline.

Run all cells from top to bottom to create `results/tables/manuscript/Manuscript_tables.xlsx`. The source CSV tables are read as prepared; their calculation and regeneration belong to `main.py`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path.cwd().resolve()
while not ((REPO / "data").is_dir() and (REPO / "code" / "scripts").is_dir()):
    if REPO == REPO.parent:
        raise FileNotFoundError("Run this notebook from the repository or one of its subfolders.")
    REPO = REPO.parent

RESULTS = REPO / "results"
TABLES = RESULTS / "tables" / "manuscript"
OUTPUT_XLSX = TABLES / "Manuscript_tables.xlsx"
required_files = [
    TABLES / "TableS1_forest_biome_additions.csv",
    TABLES / "Table3_forest_capacity_erosion.csv",
    TABLES / "Supplementary_country_level_accounts_2024.csv",
    TABLES / "Supplementary_national_top10_2024.csv",
    RESULTS / "validation" / "country_level_results_master.csv",
    RESULTS / "draws" / "ncw_draws.npz",
    RESULTS / "draws" / "gcw_draws.npz",
]
missing_files = [str(path.relative_to(REPO)) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Required prepared results are missing: " + ", ".join(missing_files))

master = pd.read_csv(RESULTS / "validation" / "country_level_results_master.csv")
source_registry = pd.read_csv(REPO / "data" / "metadata" / "source_registry.csv")
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 24)
print(f"Repository: {REPO}")
print(f"Current country-level results: {len(master)} unique UNIT_CODE records")
print(f"Prepared manuscript tables: {TABLES}")
print("Source CSV tables are read from the outputs prepared by main.py.")

Repository: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo
Current country-level results: 266 unique UNIT_CODE records
Prepared manuscript tables: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo\results\tables\manuscript
Source CSV tables are read from the outputs prepared by main.py.


## Manuscript Table 1. Biome-specific contributions to annual gross forest carbon removals

Forest areas are derived from FAO FRA 2025 and allocated to forest biomes using the current FRA-first allocation. Removal rates come from the Gibbs/GFW 2001-2023 framework after source pixels classified as mangroves are excluded. Gross additions are modeled annual additions to living forest biomass, not net forest carbon uptake.

The table below is read from the current generated manuscript output, not typed from an earlier draft.

In [2]:
table1 = pd.read_csv(TABLES / "TableS1_forest_biome_additions.csv")
table1.columns = [
    "Forest biome",
    "Forest area (Mha)",
    "Removal rate (tC ha-1 yr-1)",
    "Gross additions (GtCO2 yr-1)",
    "Global share (%)",
]
for column in table1.columns[1:]:
    table1[column] = pd.to_numeric(
        table1[column].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
print(f"Current total: {table1.iloc[-1, 1]:,.1f} Mha and {table1.iloc[-1, 3]:.3f} GtCO2/yr.")
display(table1)

Current total: 4,136.2 Mha and 14.764 GtCO2/yr.


,Forest biome,Forest area (Mha),Removal rate (tC ha-1 yr-1),Gross additions (GtCO2 yr-1),Global share (%)
0,Tropical,1852.7,0.934,6.348,43.0
1,Subtropical,436.3,1.371,2.193,14.9
2,Temperate,683.3,1.430,3.583,24.3
3,Boreal,1163.9,0.619,2.640,17.9
4,Total,4136.2,NaN,14.764,100.0


## Manuscript Table 2. Changes in national carbon-accounting balances after marine removals are included

The sample requires both GCW and BCW values and a matched 2024 fossil-fuel and industrial CO2 emissions value. A balance is positive when modeled annual removals exceed those emissions. Newly positive means non-positive with GCW and positive with NCW. The total is the number of unique study units, not a sum of separately rounded percentages.

In [3]:
balance_sample = master.loc[
    master["has_gcw"].fillna(False).astype(bool)
    & master["has_bcw"].fillna(False).astype(bool)
    & master["co2_emissions_tco2_2024"].notna()
].copy()
if balance_sample["UNIT_CODE"].duplicated().any():
    raise ValueError("The Table 2 sample must have one record per UNIT_CODE")

balance_sample["positive_gcw"] = balance_sample["gcw_accounting_balance_mean_tco2_yr"].gt(0)
balance_sample["positive_ncw"] = balance_sample["ncw_accounting_balance_mean_tco2_yr"].gt(0)
balance_sample["newly_positive"] = ~balance_sample["positive_gcw"] & balance_sample["positive_ncw"]

table2_rows = []
group_order = ["Developed economies", "Developing economies", "LDCs", "SIDS"]
for group in group_order:
    part = balance_sample.loc[balance_sample["groups"].eq(group)]
    n_units = len(part)
    newly_positive = int(part["newly_positive"].sum())
    table2_rows.append(
        {
            "Country group": group,
            "Countries and territories (n)": n_units,
            "Positive with GCW": int(part["positive_gcw"].sum()),
            "Positive with NCW": int(part["positive_ncw"].sum()),
            "Newly positive": newly_positive,
            "Newly positive (%)": 100 * newly_positive / n_units if n_units else np.nan,
        }
    )

n_total = len(balance_sample)
n_new_total = int(balance_sample["newly_positive"].sum())
table2_rows.append(
    {
        "Country group": "Total",
        "Countries and territories (n)": n_total,
        "Positive with GCW": int(balance_sample["positive_gcw"].sum()),
        "Positive with NCW": int(balance_sample["positive_ncw"].sum()),
        "Newly positive": n_new_total,
        "Newly positive (%)": 100 * n_new_total / n_total if n_total else np.nan,
    }
)
table2 = pd.DataFrame(table2_rows)
display(table2)
print(f"Current complete-case sample: {n_total} unique units; {n_new_total} become positive after marine removals are added.")

,Country group,Countries and territories (n),Positive with GCW,Positive with NCW,Newly positive,Newly positive (%)
0,Developed economies,52,10,16,6,11.538462
1,Developing economies,71,26,32,6,8.450704
2,LDCs,36,29,32,3,8.333333
3,SIDS,49,14,34,20,40.816327
4,Total,208,79,114,35,16.826923


Current complete-case sample: 208 unique units; 35 become positive after marine removals are added.


## Manuscript Table 3. Annual forest-capacity loss associated with reported deforestation, 2010-2020

Reporting coverage differs across groups. Missing deforestation reports remain missing and are not counted as zero loss. The median erosion is the unit-level annual loss of removal capacity divided by GCW. Totals are calculated from the current country-level results.

In [4]:
table3 = pd.read_csv(TABLES / "Table3_forest_capacity_erosion.csv").rename(
    columns={
        "group": "Country group",
        "reporting_units": "Reporting units",
        "positive_loss_units": "Units with positive loss",
        "lost_capacity_mtco2_yr": "Lost capacity (MtCO2 yr-1)",
        "lost_gcw_billion_usd2015_yr": "Lost GCW (billion 2015 US$ yr-1)",
        "median_erosion_pct_of_gcw": "Median erosion (% of GCW)",
    }
)
report = master.loc[master["reported_deforestation"].fillna(False).astype(bool)]
unclassified_reporters = int(report["groups"].isna().sum())
print(
    f"Current reports: {len(report)}; positive estimated losses: {int(report['loss_tco2_mean'].gt(0).sum())}; "
    f"reporting units without an economic-group label: {unclassified_reporters}."
)
display(table3)

Current reports: 139; positive estimated losses: 100; reporting units without an economic-group label: 0.


,Country group,Reporting units,Units with positive loss,Lost capacity (MtCO2 yr-1),Lost GCW (billion 2015 US$ yr-1),Median erosion (% of GCW)
0,Developed economies,47,30,2.869163,1.605506,0.019853
1,Developing economies,51,36,20.731026,11.600520,0.150520
2,LDCs,24,22,9.403865,5.262148,1.149668
3,SIDS,17,12,0.373345,0.208914,0.068612
4,Unclassified,0,0,NaN,NaN,NaN
5,Total,139,100,33.377400,18.677088,NaN


## Sources and calculation detail

This sheet links each major account component to the registered source, release, local input path and the transformation used in this workbook. FRA climatic-domain and mapped ecological-zone shares are reported as forest-biome composition; the repository does not contain a separate country-level Koppen climate classification.

In [5]:
source_specs = [
    {
        "Workstream": "Forest carbon",
        "Subcomponent": "National forest area and FRA reporting",
        "source_id": "fra_2025",
        "Calculation in this workbook": "Uses reported FRA 2025 forest area. Only mangrove area explicitly documented as already classified as Forest is removed from the forest total; Other Wooded Land is not added to Forest.",
    },
    {
        "Workstream": "Forest carbon",
        "Subcomponent": "Forest-biome or climatic-domain allocation",
        "source_id": "fra_2025",
        "Calculation in this workbook": "Uses country-reported FRA 2025 biome shares where available; otherwise uses the repository's documented ecological-map fallback.",
    },
    {
        "Workstream": "Forest carbon",
        "Subcomponent": "Map fallback for biome allocation",
        "source_id": "fao_gez_2010",
        "Calculation in this workbook": "FAO Global Ecological Zones are used in the fallback allocation where FRA country-reported shares are unavailable.",
    },
    {
        "Workstream": "Forest carbon",
        "Subcomponent": "Forest mask in biome-allocation fallback",
        "source_id": "jrc_gfc2020_v4",
        "Calculation in this workbook": "JRC Global Forest Cover 2020 mask weights ecological-zone areas in the fallback; it is not substituted for FRA 2025 national forest area.",
    },
    {
        "Workstream": "Forest carbon",
        "Subcomponent": "Gross removal rates by biome",
        "source_id": "gfw_gibbs_gross_removals",
        "Calculation in this workbook": "Mean annual 2001-2023 living-biomass additions are area-weighted within four biomes after source pixels classified as mangroves are excluded.",
    },
    {
        "Workstream": "Forest carbon",
        "Subcomponent": "External rate comparison",
        "source_id": "pan_2024_extended_tables",
        "Calculation in this workbook": "Retained as a contextual sensitivity/comparison only; Pan rates do not enter the Gibbs/GFW main estimate.",
    },
    {
        "Workstream": "Marine carbon",
        "Subcomponent": "Saltmarsh, seagrass, and source BCW account inputs",
        "source_id": "bcw_country",
        "Calculation in this workbook": "Source physical components are mapped to common UNIT_CODE values; component flux draws are aligned to the source means and Monte Carlo standard errors.",
    },
    {
        "Workstream": "Marine carbon",
        "Subcomponent": "Common social-cost and marine valuation draws",
        "source_id": "bcw_draws",
        "Calculation in this workbook": "The common GSCC draw index is used for GCW, BCW, NCW, and component-level wealth summaries.",
    },
    {
        "Workstream": "Marine carbon",
        "Subcomponent": "Current mangrove area and sequestration",
        "source_id": "gmw_2025_derived",
        "Calculation in this workbook": "Selected mangrove area by unit is valued with the BCW mangrove-rate draws and the common GSCC draws. The main country table reports total BCW, not a separate mangrove-value column; ecosystem detail is in Components.",
    },
    {
        "Workstream": "Marine carbon",
        "Subcomponent": "Open-ocean BCP",
        "source_id": "bcw_country",
        "Calculation in this workbook": "The source open-ocean BCP physical flux is retained and valued with the common GSCC draws.",
    },
    {
        "Workstream": "Economic data",
        "Subcomponent": "Population, GDP, and GDP per capita for 2024",
        "source_id": "wb_wdi_population_2024",
        "Calculation in this workbook": "Country-level population and constant-2015-US-dollar GDP are joined to UNIT_CODE for 2024 ratios and descriptive data.",
    },
    {
        "Workstream": "Economic data",
        "Subcomponent": "GDP source for 2024",
        "source_id": "wb_wdi_gdp_2024",
        "Calculation in this workbook": "GDP is in constant 2015 US dollars; ratios use the same 2024 GDP values.",
    },
    {
        "Workstream": "Economic data",
        "Subcomponent": "Fossil-fuel and industrial-process CO2 emissions for 2024",
        "source_id": "gcb_2025_national_fossil",
        "Calculation in this workbook": "The 2024 territorial series defines the emissions denominator in Table 2 and the country-level removal-to-emissions ratios.",
    },
    {
        "Workstream": "Climate finance",
        "Subcomponent": "Published reforestation action-cost curves",
        "source_id": "busch_2024_mac",
        "Calculation in this workbook": "Published curves remain separate from the provisional cell-level reconstruction and the stipulated 2% GCW allocation.",
    },
    {
        "Workstream": "Climate finance",
        "Subcomponent": "2015/2020 price conversion",
        "source_id": "bls_cpi_u_annual",
        "Calculation in this workbook": "Annual CPI-U values document conversion of Busch 2020 US dollars to 2015 US dollars in finance exports.",
    },
    {
        "Workstream": "Accounting geography",
        "Subcomponent": "Common country and territory identifiers",
        "source_id": "bcw_country",
        "Calculation in this workbook": "FRA/BCW rows are harmonised through the repository crosswalk at data/reference/territory_crosswalk.csv; the exported key is UNIT_CODE.",
    },
    {
        "Workstream": "Accounting geography",
        "Subcomponent": "Researcher-maintained group conventions",
        "source_id": "project_group_overrides",
        "Calculation in this workbook": "Explicit project assignments are stored in data/reference/economic_group_overrides.csv; they are not inferred in this notebook.",
    },
]

registry = source_registry.set_index("source_id")
source_rows = []
for spec in source_specs:
    record = dict(spec)
    source_id = record.pop("source_id")
    record["Source ID"] = source_id
    if source_id in registry.index:
        source = registry.loc[source_id]
        record.update(
            {
                "Source dataset": source["title"],
                "Provider": source["provider"],
                "Release / period": source["version_or_release"],
                "Source URL": source["url"],
                "Repository input": source["raw_target"],
            }
        )
    elif source_id == "gmw_2025_derived":
        record.update(
            {
                "Source dataset": "Global Mangrove Watch 2025 area by accounting unit",
                "Provider": "Global Mangrove Watch; area table prepared in this repository",
                "Release / period": "2025 extent",
                "Source URL": registry.loc["gmw_2025_corrected_stats", "url"],
                "Repository input": "data/derived/gmw_2025_mangrove_area_by_unit.csv",
            }
        )
    else:
        record.update(
            {
                "Source dataset": "Project-maintained classification file",
                "Provider": "Researcher-maintained repository input",
                "Release / period": "Current repository version",
                "Source URL": "",
                "Repository input": "data/reference/economic_group_overrides.csv",
            }
        )
    source_rows.append(record)

source_table = pd.DataFrame(source_rows)[
    [
        "Workstream",
        "Subcomponent",
        "Source ID",
        "Source dataset",
        "Provider",
        "Release / period",
        "Repository input",
        "Source URL",
        "Calculation in this workbook",
    ]
]
display(source_table)

,Workstream,Subcomponent,Source ID,Source dataset,Provider,Release / period,Repository input,Source URL,Calculation in this workbook
0,Forest carbon,National forest area and FRA reporting,fra_2025,Global Forest Resources Assessment 2025,FAO,FRA 2025,data/raw/environmental/forest/fra/fra2025_bulk...,https://fra-data.fao.org/api/file/bulk-downloa...,Uses reported FRA 2025 forest area. Only mangr...
1,Forest carbon,Forest-biome or climatic-domain allocation,fra_2025,Global Forest Resources Assessment 2025,FAO,FRA 2025,data/raw/environmental/forest/fra/fra2025_bulk...,https://fra-data.fao.org/api/file/bulk-downloa...,Uses country-reported FRA 2025 biome shares wh...
2,Forest carbon,Map fallback for biome allocation,fao_gez_2010,Global Ecological Zones second edition (2010),FAO,GEZ 2010,data/raw/environmental/forest/gez/gez2010.zip,https://storage.googleapis.com/fao-maps-catalo...,FAO Global Ecological Zones are used in the fa...
3,Forest carbon,Forest mask in biome-allocation fallback,jrc_gfc2020_v4,"JRC Global Forest Cover 2020 V4, overview x256...",European Commission JRC,V4 COG (modified 2026-09-14); internal overvie...,data/raw/environmental/forest/forest_mask/JRC_...,https://jeodpp.jrc.ec.europa.eu/ftp/jrc-openda...,JRC Global Forest Cover 2020 mask weights ecol...
4,Forest carbon,Gross removal rates by biome,gfw_gibbs_gross_removals,Gross forest carbon removals 2001-2023,D. A. Gibbs et al. / World Resources Institute,published model version accompanying Gibbs et ...,data/raw/environmental/forest/gfw_gibbs_2025/g...,https://doi.org/10.7910/DVN/V2ISRH,Mean annual 2001-2023 living-biomass additions...
5,Forest carbon,External rate comparison,pan_2024_extended_tables,Pan et al. 2024 Table 1 (HTML) and Extended Da...,Nature,version of record 2024-07-17,data/raw/environmental/forest/pan/nature_exten...,https://www.nature.com/articles/s41586-024-076...,Retained as a contextual sensitivity/compariso...
6,Marine carbon,"Saltmarsh, seagrass, and source BCW account in...",bcw_country,BCW final country-level data,BCW repository,commit 29ede5b09b859aa0367221d47dbc2aa1752eb627,data/raw/environmental/marine/bcw/output/count...,https://github.com/aurvl/bcw-paper-2025,Source physical components are mapped to commo...
7,Marine carbon,Common social-cost and marine valuation draws,bcw_draws,BCW GSCC and valuation draws,BCW repository,commit 29ede5b09b859aa0367221d47dbc2aa1752eb627,data/raw/environmental/marine/bcw/output/draws...,https://github.com/aurvl/bcw-paper-2025,"The common GSCC draw index is used for GCW, BC..."
8,Marine carbon,Current mangrove area and sequestration,gmw_2025_derived,Global Mangrove Watch 2025 area by accounting ...,Global Mangrove Watch; area table prepared in ...,2025 extent,data/derived/gmw_2025_mangrove_area_by_unit.csv,https://zenodo.org/records/21346457,Selected mangrove area by unit is valued with ...
9,Marine carbon,Open-ocean BCP,bcw_country,BCW final country-level data,BCW repository,commit 29ede5b09b859aa0367221d47dbc2aa1752eb627,data/raw/environmental/marine/bcw/output/count...,https://github.com/aurvl/bcw-paper-2025,The source open-ocean BCP physical flux is ret...


## Top country and territory rankings

The current prepared ranking output contains the top ten units for each listed metric. This keeps the rank definition visible in the `Metric` column instead of mixing unlike quantities into one ranking.

In [6]:
top_source = pd.read_csv(TABLES / "Supplementary_national_top10_2024.csv")
top_countries = top_source[
    ["metric", "rank_descending", "UNIT_CODE", "ISO3", "name", "region", "group", "value", "unit"]
].rename(
    columns={
        "metric": "Metric",
        "rank_descending": "Rank",
        "UNIT_CODE": "UNIT_CODE",
        "ISO3": "ISO3",
        "name": "Country or territory",
        "region": "Region",
        "group": "Economic group",
        "value": "Value",
        "unit": "Unit",
    }
)
print(f"{len(top_countries)} ranked rows across {top_countries['Metric'].nunique()} metrics.")
display(top_countries)

120 ranked rows across 12 metrics.


,Metric,Rank,UNIT_CODE,ISO3,Country or territory,Region,Economic group,Value,Unit
0,GCW,1,RUS,RUS,Russian Federation,Europe,Developed economies,1.195472e+12,US$2015/yr
1,GCW,2,BRA,BRA,Brazil,Americas,Developing economies,9.383748e+11,US$2015/yr
2,GCW,3,USA,USA,United States of America,Americas,Developed economies,8.194324e+11,US$2015/yr
3,GCW,4,CHN,CHN,"China (incl. Taiwan, Hong Kong SAR and Macao SAR)",Asia,Developing economies,6.101334e+11,US$2015/yr
4,GCW,5,CAN,CAN,Canada,Americas,Developed economies,6.093673e+11,US$2015/yr
...,...,...,...,...,...,...,...,...,...
115,Our 2% GCW allocation,6,AUS,AUS,Australia,Oceania,Developed economies,7.267759e+09,US$2015/yr
116,Our 2% GCW allocation,7,COD,COD,Democratic Republic of the Congo,Africa,LDCs,5.336977e+09,US$2015/yr
117,Our 2% GCW allocation,8,IDN,IDN,Indonesia,Asia,Developing economies,3.679786e+09,US$2015/yr
118,Our 2% GCW allocation,9,IND,IND,India,Asia,Developing economies,3.050323e+09,US$2015/yr


## Accounting-unit crosswalk

This sheet lists every current `UNIT_CODE`, its ISO code when one exists, names and geographical/group fields. It includes BCW-only and source aggregate rows as well as FRA country and territory records; `Accounting-unit type` makes those cases explicit. Availability booleans are intentionally omitted.

In [7]:
unit_crosswalk = master[
    [
        "UNIT_CODE",
        "ISO3",
        "name",
        "sovereign",
        "m49",
        "continent",
        "region",
        "fra_region",
        "fra_subregion",
        "groups",
        "unit_class",
        "bcw_codes",
    ]
].rename(
    columns={
        "name": "Country, territory, or source row",
        "sovereign": "Sovereign",
        "m49": "UN M49 code",
        "continent": "Continent",
        "region": "Region",
        "fra_region": "FRA region",
        "fra_subregion": "FRA subregion",
        "groups": "Economic group",
        "unit_class": "Accounting-unit type",
        "bcw_codes": "Mapped BCW source codes",
    }
).sort_values("UNIT_CODE").reset_index(drop=True)
print(f"Complete crosswalk: {len(unit_crosswalk)} unique UNIT_CODE records.")
display(unit_crosswalk.head(20))

Complete crosswalk: 266 unique UNIT_CODE records.


,UNIT_CODE,ISO3,"Country, territory, or source row",Sovereign,UN M49 code,Continent,Region,FRA region,FRA subregion,Economic group,Accounting-unit type,Mapped BCW source codes
0,ABW,ABW,Aruba,Netherlands,533.0,Caribbean,Caribbean,North and Central America,Caribbean,SIDS,FRA country or territory,ABW
1,AE,NaN,"Abu musa, Greater and Lesser Tunb",United Arab Emirates,NaN,Asia,Asia,NaN,NaN,Developing economies,BCW-only accounting unit,AE
2,AFG,AFG,Afghanistan,Afghanistan,4.0,Asia,Asia,Asia,Western and Central Asia,LDCs,FRA country or territory,AFG
3,AGO,AGO,Angola,Angola,24.0,Africa,Africa,Africa,Eastern and Southern Africa,LDCs,FRA country or territory,AGO
4,AIA,AIA,Anguilla,United Kingdom,660.0,Caribbean,Caribbean,North and Central America,Caribbean,SIDS,FRA country or territory,AIA
5,ALB,ALB,Albania,Albania,8.0,Europe,Europe,Europe,NaN,Developed economies,FRA country or territory,ALB
6,AND,AND,Andorra,Andorra,20.0,Europe,Europe,Europe,NaN,Developed economies,FRA country or territory,AND
7,ARE,ARE,United Arab Emirates,United Arab Emirates,784.0,Asia,Asia,Asia,Western and Central Asia,Developing economies,FRA country or territory,ARE
8,ARG,ARG,Argentina,Argentina,32.0,Americas,Americas,South America,NaN,Developing economies,FRA country or territory,ARG
9,ARM,ARM,Armenia,Armenia,51.0,Asia,Asia,Asia,Western and Central Asia,Developing economies,FRA country or territory,ARM


## Prepare component-level sequestration and wealth summaries

Means, sample standard deviations, and medians are calculated from the existing 20,000 aligned draws. Forest biome physical additions use fixed area-by-biome and mean rate inputs; their spatial pixel spread is kept as a separate rate field, not relabelled as draw uncertainty. Marine ecosystem components are aligned to the published component means and Monte Carlo standard errors, mapped to `UNIT_CODE`, and valued with the common GSCC draws.

In [8]:
CARBON_TO_CO2 = 44.0 / 12.0
N_DRAWS = 20000
unit_codes = master["UNIT_CODE"].astype(str).tolist()

def reindex_draw_rows(source_codes, draw_matrix, target_codes):
    source_codes = pd.Index(source_codes.astype(str))
    if source_codes.duplicated().any():
        raise ValueError("Draw archive contains duplicate unit identifiers")
    positions = {code: i for i, code in enumerate(source_codes)}
    result = np.full((len(target_codes), draw_matrix.shape[1]), np.nan, dtype=np.float64)
    for i, code in enumerate(target_codes):
        source_i = positions.get(str(code))
        if source_i is not None:
            result[i] = np.asarray(draw_matrix[source_i], dtype=np.float64)
    return result

def summarise_draw_matrix(draw_matrix, codes, prefix):
    values = np.asarray(draw_matrix, dtype=np.float64)
    result = np.full((values.shape[0], 3), np.nan, dtype=np.float64)
    valid_rows = np.isfinite(values).any(axis=1)
    valid = values[valid_rows]
    if len(valid):
        result[valid_rows, 0] = np.nanmean(valid, axis=1)
        result[valid_rows, 1] = np.nanstd(valid, axis=1, ddof=1)
        result[valid_rows, 2] = np.nanmedian(valid, axis=1)
    return pd.DataFrame(
        {
            "UNIT_CODE": codes,
            f"{prefix} mean": result[:, 0],
            f"{prefix} SD": result[:, 1],
            f"{prefix} median": result[:, 2],
        }
    )

def sum_available_draw_matrices(matrices):
    total = np.zeros_like(np.asarray(matrices[0], dtype=np.float64))
    has_component = np.zeros(total.shape[0], dtype=bool)
    for matrix in matrices:
        values = np.asarray(matrix, dtype=np.float64)
        for row_index, row in enumerate(values):
            finite = np.isfinite(row)
            if finite.any():
                total[row_index, finite] += row[finite]
                has_component[row_index] = True
    total[~has_component] = np.nan
    return total

with np.load(RESULTS / "draws" / "ncw_draws.npz", allow_pickle=True) as archive:
    account_codes = archive["iso3"].astype(str).copy()
    gcw_draws = reindex_draw_rows(account_codes, archive["gcw_usd2015_yr"], unit_codes)
    bcw_draws = reindex_draw_rows(account_codes, archive["bcw_usd2015_yr"], unit_codes)
    ncw_draws = reindex_draw_rows(account_codes, archive["ncw_usd2015_yr"], unit_codes)
    marine_flux_draws = reindex_draw_rows(account_codes, archive["bcw_sequestration_tco2_yr"], unit_codes)
    mangrove_flux_draws = reindex_draw_rows(account_codes, archive["selected_mangrove_tco2_yr"], unit_codes)
    saltmarsh_flux_draws = reindex_draw_rows(account_codes, archive["saltmarsh_tco2_yr"], unit_codes)
    seagrass_flux_draws = reindex_draw_rows(account_codes, archive["seagrass_tco2_yr"], unit_codes)
    ocean_flux_draws = reindex_draw_rows(account_codes, archive["bcp_tco2_yr"], unit_codes)

with np.load(RESULTS / "draws" / "gcw_draws.npz", allow_pickle=True) as archive:
    gscc_draws = archive["gscc"].astype(np.float64).copy()
if gscc_draws.size != N_DRAWS:
    raise ValueError(f"Expected {N_DRAWS} common GSCC draws; found {gscc_draws.size}")

with np.load(REPO / "data" / "derived" / "draws" / "forest_uptake_gfw.npz", allow_pickle=True) as archive:
    forest_codes = archive["iso3"].astype(str).copy()
    forest_flux_draws = reindex_draw_rows(
        forest_codes,
        archive["documented_overlap_removed"],
        unit_codes,
    )
forest_value_check = forest_flux_draws * gscc_draws[None, :]
valid_forest = np.isfinite(forest_flux_draws).any(axis=1)
if not np.allclose(forest_value_check[valid_forest], gcw_draws[valid_forest], rtol=5e-6, atol=1.0):
    raise ValueError('Forest physical draws times GSCC do not reproduce the GCW account')

selected_area = pd.to_numeric(master["mangrove_area_selected_ha"], errors="coerce").to_numpy(float)
selected_area_rows = np.isfinite(selected_area)
bcw_formula_draws = marine_flux_draws * gscc_draws[None, :]
valid_bcw_formula = selected_area_rows[:, None] & np.isfinite(bcw_formula_draws) & np.isfinite(bcw_draws)
if valid_bcw_formula.any():
    bcw_relative_error = np.abs(bcw_formula_draws[valid_bcw_formula] - bcw_draws[valid_bcw_formula]) / np.maximum(
        np.abs(bcw_draws[valid_bcw_formula]), 1.0
    )
    MAX_BCW_FORMULA_RELATIVE_DIFFERENCE = float(bcw_relative_error.max())
    if MAX_BCW_FORMULA_RELATIVE_DIFFERENCE > 1e-5:
        raise ValueError(
            "Direct total BCW flux times GSCC does not reproduce the updated BCW account; "
            f"maximum relative difference is {MAX_BCW_FORMULA_RELATIVE_DIFFERENCE:.6g}"
        )
else:
    MAX_BCW_FORMULA_RELATIVE_DIFFERENCE = float("nan")

# Each shared draw combines the available forest and total blue-account physical fluxes.
total_physical_draws = sum_available_draw_matrices([forest_flux_draws, marine_flux_draws])
account_matrices = {
    "GCW": gcw_draws,
    "BCW": bcw_draws,
    "NCW": ncw_draws,
}
for account_name, draw_matrix in account_matrices.items():
    summary = summarise_draw_matrix(draw_matrix, unit_codes, account_name)
    mean_column = f"{account_name} mean"
    reference_column = {
        "GCW": "gcw_usd2015_yr_mean",
        "BCW": "bcw_usd2015_yr_mean",
        "NCW": "ncw_usd2015_yr_mean",
    }[account_name]
    check = master[["UNIT_CODE", reference_column]].merge(
        summary[["UNIT_CODE", mean_column]], on="UNIT_CODE", validate="one_to_one"
    ).dropna()
    if len(check) and not np.allclose(
        check[reference_column], check[mean_column], rtol=5e-4, atol=1.0
    ):
        raise ValueError(f"The {account_name} draw summaries do not match the canonical country table")

print(
    f"Draw check: {N_DRAWS:,} shared draws. Direct BCW formula: total physical flux × 44/12 × GSCC; "
    f"maximum relative difference {MAX_BCW_FORMULA_RELATIVE_DIFFERENCE:.6g}."
)


Draw check: 20,000 shared draws. Direct BCW formula: total physical flux × 44/12 × GSCC; maximum relative difference 5.95838e-08.


## Ecosystem and account detail by accounting unit

One row represents one `UNIT_CODE` and one forest biome, blue-carbon ecosystem, or account total. Forest biome wealth is the biome-specific physical addition valued with the common GSCC draws. Current mangrove ecosystem detail uses the selected area under the GMW 2025 > FRA 2025 > GMW 2024 > GMW 2023 > Hilmi-reported-area hierarchy and the current BCW mangrove-rate draws. Total GCW, BCW, and NCW rows use the canonical account draws.

In [9]:
component_identity = master[
    ["UNIT_CODE", "ISO3", "name", "region", "groups", "unit_class"]
].rename(
    columns={
        "name": "Country or territory",
        "region": "Region",
        "groups": "Economic group",
        "unit_class": "Accounting-unit type",
    }
).reset_index(drop=True)

def component_record_rows(
    domain,
    component,
    physical_draw_matrix,
    wealth_draw_matrix,
    area_values=None,
    area_unit="",
    biome_share=None,
    rate_mean=None,
    rate_spatial_sd=None,
    calculation_note="",
):
    physical_stats = summarise_draw_matrix(
        physical_draw_matrix, unit_codes, "physical"
    ).drop(columns="UNIT_CODE")
    if wealth_draw_matrix is None:
        wealth_stats = pd.DataFrame({
            "wealth mean": np.nan,
            "wealth SD": np.nan,
            "wealth median": np.nan,
        }, index=range(len(unit_codes)))
    else:
        wealth_stats = summarise_draw_matrix(
            wealth_draw_matrix, unit_codes, "wealth"
        ).drop(columns="UNIT_CODE")
    rows = component_identity.copy()
    rows["Component domain"] = domain
    rows["Component"] = component
    rows["Area"] = np.nan if area_values is None else np.asarray(area_values, dtype=float)
    rows["Area unit"] = area_unit
    rows["Forest share (%)"] = np.nan if biome_share is None else np.asarray(biome_share, dtype=float)
    rows["Mean removal rate (tC ha-1 yr-1)"] = np.nan if rate_mean is None else np.asarray(rate_mean, dtype=float)
    rows["Pixel spatial SD of rate (tC ha-1 yr-1)"] = np.nan if rate_spatial_sd is None else np.asarray(rate_spatial_sd, dtype=float)
    rows["Physical flux mean (tCO2 yr-1)"] = physical_stats["physical mean"].to_numpy()
    rows["Physical flux SD (tCO2 yr-1)"] = physical_stats["physical SD"].to_numpy()
    rows["Physical flux median (tCO2 yr-1)"] = physical_stats["physical median"].to_numpy()
    rows["Wealth mean (2015 US$ yr-1)"] = wealth_stats["wealth mean"].to_numpy()
    rows["Wealth SD (2015 US$ yr-1)"] = wealth_stats["wealth SD"].to_numpy()
    rows["Wealth median (2015 US$ yr-1)"] = wealth_stats["wealth median"].to_numpy()
    rows["Calculation note"] = calculation_note
    return rows

component_frames = []
is_fra = master["in_fra"].fillna(False).astype(bool).to_numpy()
for biome in ["tropical", "subtropical", "temperate", "boreal"]:
    area = master[f"forest_area_allocated_{biome}_ha"].to_numpy(dtype=float, copy=True)
    area[is_fra & ~np.isfinite(area)] = 0.0
    share = master[f"forest_biome_share_{biome}"].to_numpy(dtype=float, copy=True) * 100.0
    share[is_fra & ~np.isfinite(share)] = 0.0
    rate = master[f"forest_rate_{biome}_tC_ha_yr"].to_numpy(float)
    spatial_sd = master[f"forest_rate_{biome}_spatial_sd_tC_ha_yr"].to_numpy(float)
    additions = master[f"forest_additions_{biome}_tCO2_yr"].to_numpy(dtype=float, copy=True)
    additions[is_fra & ~np.isfinite(additions)] = 0.0
    physical_draws = np.full((len(unit_codes), N_DRAWS), np.nan, dtype=np.float64)
    valid = is_fra & np.isfinite(additions)
    physical_draws[valid] = additions[valid, None]
    wealth_draws = physical_draws * gscc_draws[None, :]
    component_frames.append(
        component_record_rows(
            "Forest biome",
            biome.title(),
            physical_draws,
            wealth_draws,
            area_values=area,
            area_unit="ha",
            biome_share=share,
            rate_mean=rate,
            rate_spatial_sd=spatial_sd,
            calculation_note="Physical additions are fixed area x biome mean rate. The rate pixel spatial SD is shown separately and is not a confidence interval.",
        )
    )

salt_area = master["saltmarsh_area_km2_hilmi_source"].to_numpy(float)
sea_area = master["seagrass_area_km2_hilmi_source"].to_numpy(float)
mangrove_area = master["mangrove_area_selected_ha"].to_numpy(float)
eez_area = master["bcw_source_eez_area_km2"].to_numpy(float)
component_frames.extend(
    [
        component_record_rows("Blue-carbon ecosystem", "Saltmarsh", saltmarsh_flux_draws, None, salt_area, "km2", calculation_note="Source BCW physical component draws, mapped to UNIT_CODE and converted from tC to tCO2."),
        component_record_rows("Blue-carbon ecosystem", "Seagrass", seagrass_flux_draws, None, sea_area, "km2", calculation_note="Source BCW physical component draws, mapped to UNIT_CODE and converted from tC to tCO2."),
        component_record_rows("Blue-carbon ecosystem", "Mangrove (selected area source)", mangrove_flux_draws, None, mangrove_area, "ha", calculation_note="Selected mangrove area and the physical mangrove-sediment flux calculated from that area and the BCW rate draws; its monetary effect is included only in total BCW."),
        component_record_rows("Open-ocean carbon", "Open-ocean BCP", ocean_flux_draws, None, eez_area, "km2 EEZ", calculation_note="Source open-ocean BCP flux is held fixed; the monetary distribution propagates the common GSCC draws."),
        component_record_rows("Account total", "Total GCW", forest_flux_draws, gcw_draws, calculation_note="Canonical forest gross additions and GCW account draws."),
        component_record_rows("Account total", "Total BCW", marine_flux_draws, bcw_draws, calculation_note="Total BCW is calculated directly from selected coastal areas and rates plus the Hilmi-reported BCP flux, then multiplied by 44/12 and the shared GSCC draw."),
        component_record_rows("Account total", "Total NCW", total_physical_draws, ncw_draws, calculation_note="Canonical NCW draws; available GCW and BCW components are summed, and rows with neither component remain blank."),
    ]
)
component_detail = pd.concat(component_frames, ignore_index=True)
print(f"Component detail: {len(component_detail):,} rows x {len(component_detail.columns)} columns.")
display(component_detail.head(16))

Component detail: 2,926 rows x 20 columns.


,UNIT_CODE,ISO3,Country or territory,Region,Economic group,Accounting-unit type,Component domain,Component,Area,Area unit,Forest share (%),Mean removal rate (tC ha-1 yr-1),Pixel spatial SD of rate (tC ha-1 yr-1),Physical flux mean (tCO2 yr-1),Physical flux SD (tCO2 yr-1),Physical flux median (tCO2 yr-1),Wealth mean (2015 US$ yr-1),Wealth SD (2015 US$ yr-1),Wealth median (2015 US$ yr-1),Calculation note
0,ABW,ABW,Aruba,Caribbean,SIDS,FRA country or territory,Forest biome,Tropical,490.0,ha,100.0,0.934399,0.926969,1.678803e+03,2.273794e-13,1.678803e+03,9.394125e+05,4.509171e+05,8.506860e+05,Physical additions are fixed area x biome mean...
1,AE,NaN,"Abu musa, Greater and Lesser Tunb",Asia,Developing economies,BCW-only accounting unit,Forest biome,Tropical,NaN,ha,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Physical additions are fixed area x biome mean...
2,AFG,AFG,Afghanistan,Asia,LDCs,FRA country or territory,Forest biome,Tropical,0.0,ha,0.0,0.934399,0.926969,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Physical additions are fixed area x biome mean...
3,AGO,AGO,Angola,Africa,LDCs,FRA country or territory,Forest biome,Tropical,63262330.0,ha,100.0,0.934399,0.926969,2.167449e+08,5.960613e-08,2.167449e+08,1.212845e+11,5.821646e+10,1.098293e+11,Physical additions are fixed area x biome mean...
4,AIA,AIA,Anguilla,Caribbean,SIDS,FRA country or territory,Forest biome,Tropical,5500.0,ha,100.0,0.934399,0.926969,1.884370e+04,3.638070e-12,1.884370e+04,1.054443e+07,5.061314e+06,9.548517e+06,Physical additions are fixed area x biome mean...
5,ALB,ALB,Albania,Europe,Developed economies,FRA country or territory,Forest biome,Tropical,0.0,ha,0.0,0.934399,0.926969,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Physical additions are fixed area x biome mean...
6,AND,AND,Andorra,Europe,Developed economies,FRA country or territory,Forest biome,Tropical,0.0,ha,0.0,0.934399,0.926969,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Physical additions are fixed area x biome mean...
7,ARE,ARE,United Arab Emirates,Asia,Developing economies,FRA country or territory,Forest biome,Tropical,324580.0,ha,100.0,0.934399,0.926969,1.112053e+06,0.000000e+00,1.112053e+06,6.222745e+08,2.986912e+08,5.635014e+08,Physical additions are fixed area x biome mean...
8,ARG,ARG,Argentina,Americas,Developing economies,FRA country or territory,Forest biome,Tropical,34024570.0,ha,73.0,0.934399,0.926969,1.165725e+08,1.490153e-08,1.165725e+08,6.523083e+10,3.131074e+10,5.906985e+10,Physical additions are fixed area x biome mean...
9,ARM,ARM,Armenia,Asia,Developing economies,FRA country or territory,Forest biome,Tropical,0.0,ha,0.0,0.934399,0.926969,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Physical additions are fixed area x biome mean...


## Country-level NCW table

This is the primary wide country-level sheet. It combines identifiers, 2024 economic and emissions data, FRA forest area and biome composition, marine ecosystem areas, physical removals, account wealth, and derived ratios. It deliberately excludes the internal `has_gcw` / `has_bcw` availability flags and any standalone BCW mangrove wealth column. Component-level ecosystem wealth is provided in the Components sheet.

Blank account values stay blank when neither account component is available. Where one account component exists, the canonical NCW calculation sums the available component and treats the unavailable component as zero; the resulting partial NCW is described in Read me.

In [10]:
forest_summary = summarise_draw_matrix(forest_flux_draws, unit_codes, "Forest additions (tCO2 yr-1)")
marine_summary = summarise_draw_matrix(marine_flux_draws, unit_codes, "Marine removals (tCO2 yr-1)")
combined_summary = summarise_draw_matrix(total_physical_draws, unit_codes, "Combined removals (tCO2 yr-1)")
gcw_summary = summarise_draw_matrix(gcw_draws, unit_codes, "GCW (2015 US$ yr-1)")
bcw_summary = summarise_draw_matrix(bcw_draws, unit_codes, "BCW (2015 US$ yr-1)")
ncw_summary = summarise_draw_matrix(ncw_draws, unit_codes, "NCW (2015 US$ yr-1)")

def divide_draws_by_unit(draw_matrix, denominator, multiplier=1.0):
    draw_values = np.asarray(draw_matrix, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    result = np.full(draw_values.shape, np.nan, dtype=np.float64)
    if denominator.ndim == 1 and denominator.shape[0] == draw_values.shape[0]:
        valid = np.isfinite(denominator) & (denominator != 0)
        np.divide(
            draw_values,
            denominator[:, None],
            out=result,
            where=valid[:, None],
        )
    elif denominator.shape == draw_values.shape:
        valid = np.isfinite(denominator) & (denominator != 0)
        np.divide(draw_values, denominator, out=result, where=valid)
    else:
        raise ValueError(
            f"Denominator shape {denominator.shape} does not match draw matrix {draw_values.shape}."
        )
    result *= multiplier
    return result

gdp_2024 = master["gdp_2015usd_2024"].to_numpy(float)
population_2024 = master["population_2024"].to_numpy(float)
emissions_2024 = master["co2_emissions_tco2_2024"].to_numpy(float)
ratio_matrices = [
    summarise_draw_matrix(divide_draws_by_unit(gcw_draws, gdp_2024, 100), unit_codes, "GCW / 2024 GDP (%)"),
    summarise_draw_matrix(divide_draws_by_unit(bcw_draws, gdp_2024, 100), unit_codes, "BCW / 2024 GDP (%)"),
    summarise_draw_matrix(divide_draws_by_unit(ncw_draws, gdp_2024, 100), unit_codes, "NCW / 2024 GDP (%)"),
    summarise_draw_matrix(divide_draws_by_unit(ncw_draws, population_2024), unit_codes, "NCW per capita (2015 US$/person)"),
    summarise_draw_matrix(divide_draws_by_unit(gcw_draws, ncw_draws, 100), unit_codes, "GCW share of NCW (%)"),
    summarise_draw_matrix(divide_draws_by_unit(forest_flux_draws, emissions_2024), unit_codes, "Forest removals / 2024 emissions (ratio)"),
    summarise_draw_matrix(divide_draws_by_unit(total_physical_draws, emissions_2024), unit_codes, "Combined removals / 2024 emissions (ratio)"),
]

biome_shares = pd.read_csv(REPO / "data" / "derived" / "biome_shares.csv")
source_names = {
    "fra_reported": "FRA 2025 reported",
    "gez_forest_mask": "FAO GEZ + JRC forest-mask fallback",
    "gez_land": "FAO GEZ land-area fallback",
}
share_basis = (
    biome_shares.groupby("iso3")["source"]
    .agg(lambda values: "; ".join(sorted({source_names.get(v, v) for v in values.dropna()})))
    .rename("Biome-share basis")
    .reset_index()
    .rename(columns={"iso3": "UNIT_CODE"})
)

country_level_ncw = master[
    [
        "UNIT_CODE", "ISO3", "name", "sovereign", "m49", "region", "fra_region", "fra_subregion", "groups", "unit_class",
        "population_2024", "gdp_2015usd_2024", "gdp_per_capita_2024", "co2_emissions_tco2_2024",
        "fra_total_forest_area_2025_ha", "forest_area_allocated_total_ha",
        "forest_area_allocated_tropical_ha", "forest_area_allocated_subtropical_ha", "forest_area_allocated_temperate_ha", "forest_area_allocated_boreal_ha",
        "forest_biome_share_tropical", "forest_biome_share_subtropical", "forest_biome_share_temperate", "forest_biome_share_boreal",
        "saltmarsh_area_km2_hilmi_source", "seagrass_area_km2_hilmi_source", "mangrove_area_selected_ha", "bcw_source_eez_area_km2",
    ]
].rename(
    columns={
        "name": "Country or territory",
        "sovereign": "Sovereign",
        "m49": "UN M49 code",
        "region": "Region",
        "fra_region": "FRA region",
        "fra_subregion": "FRA subregion",
        "groups": "Economic group",
        "unit_class": "Accounting-unit type",
        "population_2024": "Population 2024 (persons)",
        "gdp_2015usd_2024": "GDP 2024 (2015 US$)",
        "gdp_per_capita_2024": "GDP per capita 2024 (2015 US$/person)",
        "co2_emissions_tco2_2024": "Fossil and industrial CO2 emissions 2024 (tCO2 yr-1)",
        "fra_total_forest_area_2025_ha": "FRA 2025 forest area reported (ha)",
        "forest_area_allocated_total_ha": "Forest area used in GCW after documented overlap removal (ha)",
        "forest_area_allocated_tropical_ha": "Tropical forest area used (ha)",
        "forest_area_allocated_subtropical_ha": "Subtropical forest area used (ha)",
        "forest_area_allocated_temperate_ha": "Temperate forest area used (ha)",
        "forest_area_allocated_boreal_ha": "Boreal forest area used (ha)",
        "forest_biome_share_tropical": "Tropical forest share (%)",
        "forest_biome_share_subtropical": "Subtropical forest share (%)",
        "forest_biome_share_temperate": "Temperate forest share (%)",
        "forest_biome_share_boreal": "Boreal forest share (%)",
        "saltmarsh_area_km2_hilmi_source": "Saltmarsh area in source account (km2)",
        "seagrass_area_km2_hilmi_source": "Seagrass area in source account (km2)",
        "mangrove_area_selected_ha": "Selected mangrove area used (ha)",
        "bcw_source_eez_area_km2": "Source EEZ area (km2)",
    }
)
for share_column in [
    "Tropical forest share (%)",
    "Subtropical forest share (%)",
    "Temperate forest share (%)",
    "Boreal forest share (%)",
]:
    country_level_ncw[share_column] *= 100.0
country_level_ncw = country_level_ncw.merge(share_basis, on="UNIT_CODE", how="left", validate="one_to_one")

for summary in [forest_summary, marine_summary, combined_summary, gcw_summary, bcw_summary, ncw_summary, *ratio_matrices]:
    country_level_ncw = country_level_ncw.merge(summary, on="UNIT_CODE", how="left", validate="one_to_one")

forbidden_columns = [
    column for column in country_level_ncw.columns
    if column.lower().startswith("has_gcw")
    or column.lower().startswith("has_bcw")
    or "bcw_mangrove" in column.lower()
]
if forbidden_columns:
    raise ValueError(f"Internal account-availability or standalone BCW-mangrove fields leaked into the primary table: {forbidden_columns}")
print(f"Primary country-level table: {len(country_level_ncw)} rows x {len(country_level_ncw.columns)} columns.")
display(country_level_ncw.head(15))

Primary country-level table: 266 rows x 68 columns.


,UNIT_CODE,ISO3,Country or territory,Sovereign,UN M49 code,Region,FRA region,FRA subregion,Economic group,Accounting-unit type,Population 2024 (persons),GDP 2024 (2015 US$),...,NCW per capita (2015 US$/person) mean,NCW per capita (2015 US$/person) SD,NCW per capita (2015 US$/person) median,GCW share of NCW (%) mean,GCW share of NCW (%) SD,GCW share of NCW (%) median,Forest removals / 2024 emissions (ratio) mean,Forest removals / 2024 emissions (ratio) SD,Forest removals / 2024 emissions (ratio) median,Combined removals / 2024 emissions (ratio) mean,Combined removals / 2024 emissions (ratio) SD,Combined removals / 2024 emissions (ratio) median
0,ABW,ABW,Aruba,Netherlands,533.0,Caribbean,North and Central America,Caribbean,SIDS,FRA country or territory,107995.0,3.665237e+09,...,150.348573,153.123361,106.792477,8.857710,5.837239e+00,7.435737,0.001824,4.336917e-19,0.001824,0.031490,2.581982e-02,0.024524
1,AE,NaN,"Abu musa, Greater and Lesser Tunb",United Arab Emirates,NaN,Asia,NaN,NaN,Developing economies,BCW-only accounting unit,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,AFG,Afghanistan,Afghanistan,4.0,Asia,Asia,Western and Central Asia,LDCs,FRA country or territory,42647492.0,1.596623e+10,...,79.861416,38.333401,72.318592,100.000000,0.000000e+00,100.000000,0.562219,1.110251e-16,0.562219,0.562219,1.110251e-16,0.562219
3,AGO,AGO,Angola,Angola,24.0,Africa,Africa,Eastern and Southern Africa,LDCs,FRA country or territory,37885849.0,1.060271e+11,...,3861.096938,1853.364723,3497.487886,82.912209,1.540641e-01,82.951514,9.704885,3.552802e-15,9.704885,11.705053,2.191158e-02,11.699466
4,AIA,AIA,Anguilla,United Kingdom,660.0,Caribbean,North and Central America,Caribbean,SIDS,FRA country or territory,NaN,NaN,...,NaN,NaN,NaN,65.512156,1.511430e+01,67.280027,0.127249,2.775627e-17,0.127249,0.208925,7.276848e-02,0.189133
5,ALB,ALB,Albania,Albania,8.0,Europe,Europe,NaN,Developed economies,FRA country or territory,2377128.0,1.556837e+10,...,1214.907152,583.267138,1101.515021,93.228871,5.546903e-01,93.381495,1.082561,2.220502e-16,1.082561,1.161229,7.110442e-03,1.159289
6,AND,AND,Andorra,Andorra,20.0,Europe,Europe,NaN,Developed economies,FRA country or territory,81938.0,3.377811e+09,...,652.451234,313.175951,590.827894,100.000000,0.000000e+00,100.000000,0.224974,5.551254e-17,0.224974,0.224974,5.551254e-17,0.224974
7,ARE,ARE,United Arab Emirates,United Arab Emirates,784.0,Asia,Asia,Western and Central Asia,Developing economies,FRA country or territory,10986400.0,4.570937e+11,...,103.265563,64.579080,87.974065,59.435009,1.444844e+01,60.705527,0.005010,0.000000e+00,0.005010,0.009131,3.323840e-03,0.008252
8,ARG,ARG,Argentina,Argentina,32.0,Americas,South America,NaN,Developing economies,FRA country or territory,45696159.0,5.837388e+11,...,2688.224518,1290.345120,2434.324572,82.190570,5.328436e-07,82.190570,1.054785,0.000000e+00,1.054785,1.283341,2.220502e-16,1.283341
9,ARM,ARM,Armenia,Armenia,51.0,Asia,Asia,Western and Central Asia,Developing economies,FRA country or territory,3033500.0,1.631513e+10,...,314.981501,151.190810,285.231829,100.000000,0.000000e+00,100.000000,0.229796,2.775627e-17,0.229796,0.229796,2.775627e-17,0.229796


## Values to review against the figures supplied in the request

The tables are rebuilt from current repository outputs. This check sheet keeps numerical differences from the supplied example visible for review before manuscript values are copied.

In [11]:
def table_value(frame, row_label, column):
    selected = frame.loc[frame.iloc[:, 0].eq(row_label), column]
    return selected.iloc[0] if len(selected) else np.nan

current_t1_area = float(table1.loc[table1["Forest biome"].eq("Total"), "Forest area (Mha)"].iloc[0])
current_t1_additions = float(table1.loc[table1["Forest biome"].eq("Total"), "Gross additions (GtCO2 yr-1)"].iloc[0])
current_t2_developing_n = int(table2.loc[table2["Country group"].eq("Developing economies"), "Countries and territories (n)"].iloc[0])
current_t2_new = int(table2.loc[table2["Country group"].eq("Total"), "Newly positive"].iloc[0])
current_t3_unclassified = int(table3.loc[table3["Country group"].eq("Unclassified"), "Reporting units"].iloc[0])
current_t3_capacity = float(table3.loc[table3["Country group"].eq("Total"), "Lost capacity (MtCO2 yr-1)"].iloc[0])
current_t3_gcw = float(table3.loc[table3["Country group"].eq("Total"), "Lost GCW (billion 2015 US$ yr-1)"].iloc[0])

review_checks = pd.DataFrame(
    [
        ["Table 1 total forest area (Mha)", 4140.0, current_t1_area],
        ["Table 1 gross additions (GtCO2/yr)", 14.733, current_t1_additions],
        ["Table 2 unique sample size", 162, len(balance_sample)],
        ["Table 2 Developing economies sample size", 56, current_t2_developing_n],
        ["Table 2 newly positive units", 35, current_t2_new],
        ["Table 3 reporting units without group classification", 2, current_t3_unclassified],
        ["Table 3 lost capacity (MtCO2/yr)", 34.83, current_t3_capacity],
        ["Table 3 lost GCW (billion 2015 US$/yr)", 19.49, current_t3_gcw],
    ],
    columns=["Check", "Value in supplied table", "Value from current repository outputs"],
)
review_checks["Difference (current minus supplied)"] = (
    review_checks["Value from current repository outputs"] - review_checks["Value in supplied table"]
)
review_checks["Status"] = np.where(
    np.isclose(review_checks["Difference (current minus supplied)"], 0, atol=0.0005, rtol=0),
    "Matches within display precision",
    "Review: differs",
)
display(review_checks)

,Check,Value in supplied table,Value from current repository outputs,Difference (current minus supplied),Status
0,Table 1 total forest area (Mha),4140.000,4136.200000,-3.800000,Review: differs
1,Table 1 gross additions (GtCO2/yr),14.733,14.764000,0.031000,Review: differs
2,Table 2 unique sample size,162.000,208.000000,46.000000,Review: differs
3,Table 2 Developing economies sample size,56.000,71.000000,15.000000,Review: differs
4,Table 2 newly positive units,35.000,35.000000,0.000000,Matches within display precision
5,Table 3 reporting units without group classifi...,2.000,0.000000,-2.000000,Review: differs
6,Table 3 lost capacity (MtCO2/yr),34.830,33.377400,-1.452600,Review: differs
7,Table 3 lost GCW (billion 2015 US$/yr),19.490,18.677088,-0.812912,Review: differs


## Workbook notes and field guide

This sheet defines the workbook tabs, samples, draw summaries, climate/biome fields and missing-value convention. It also records that the supplied examples and the current repository outputs are not identical where the Review checks sheet says so.

In [12]:
readme = pd.DataFrame(
    [
        ["Table 1 Forest biomes", "Global forest area, biome-specific Gibbs/GFW gross additions, and each biome's share of total additions.", "Four biomes plus total; current generated manuscript table."],
        ["Table 2 Balance shifts", "Counts of units with positive annual balance before and after marine removals are included.", f"{len(balance_sample)} units with both accounts and matched 2024 emissions."],
        ["Table 3 Forest loss", "Group-level reported deforestation and associated annual capacity loss.", f"{len(report)} FRA reporters; only current report values are used."],
        ["Sources & methods", "Source registry entries and component-level transformations used in this workbook.", "Source URLs and repository input paths are included where registered."],
        ["Top 10 by metric", "Top ten units for each metric in the current national ranking output.", f"{len(top_countries)} rows across {top_countries['Metric'].nunique()} metrics."],
        ["Unit crosswalk", "All study UNIT_CODE values, ISO codes, names, and geography/group fields.", f"{len(unit_crosswalk)} rows, including non-country aggregate source rows labelled by type."],
        ["Components", "Long-format physical-removal and monetary-wealth summaries by forest biome, marine ecosystem, and account total.", f"Stats use {N_DRAWS:,} aligned draws where available; source components are validated against total BCW."],
        ["Country-level NCW", "Primary wide country/account table with economy, climate/biome, physical removals, wealth, and ratios.", "No has_gcw / has_bcw flags or standalone BCW mangrove wealth column."],
        ["Review checks", "Differences between the numbers in the supplied table examples and current repository-generated results.", "Review these rows before transferring numbers into the manuscript."],
        ["Statistics", "Mean, sample SD (ddof=1), and median are computed draw by draw using the shared monetary draws.", "These monetary draw summaries are not confidence intervals for the 2001-2023 biome removal rates."],
        ["Forest biome fields", "Tropical, subtropical, temperate, and boreal shares describe the current forest-area allocation.", "FRA-reported shares are preferred; map fallbacks are identified in Biome-share basis."],
        ["Climate scope", "The repository provides FAO forest-biome/climatic-domain shares for country-level forest allocation.", "No separate country-level Koppen classification is present in the current data build."],
        ["NCW missing values", "NCW sums the available GCW and BCW component values; an unavailable component contributes zero when the other exists.", "If both components are unavailable, NCW remains blank. Availability flags are omitted from the primary table by request."],
        ["Ecosystem wealth", "Saltmarsh, seagrass, selected mangrove area, and open-ocean BCP detail is in Components.", "The BCW total is calculated from the selected mangrove area, Hilmi-reported seagrass and saltmarsh areas, ecosystem removal rates, open-ocean BCP, and the shared GSCC draw."],
        ["Workbook path", str(OUTPUT_XLSX.relative_to(REPO)), "Written by the final notebook cell after every table has been prepared."],
    ],
    columns=["Workbook sheet or field", "What it contains", "Scope or interpretation"],
)
display(readme)

,Workbook sheet or field,What it contains,Scope or interpretation
0,Table 1 Forest biomes,"Global forest area, biome-specific Gibbs/GFW g...",Four biomes plus total; current generated manu...
1,Table 2 Balance shifts,Counts of units with positive annual balance b...,208 units with both accounts and matched 2024 ...
2,Table 3 Forest loss,Group-level reported deforestation and associa...,139 FRA reporters; only current report values ...
3,Sources & methods,Source registry entries and component-level tr...,Source URLs and repository input paths are inc...
4,Top 10 by metric,Top ten units for each metric in the current n...,120 rows across 12 metrics.
5,Unit crosswalk,"All study UNIT_CODE values, ISO codes, names, ...","266 rows, including non-country aggregate sour..."
6,Components,Long-format physical-removal and monetary-weal...,"Stats use 20,000 aligned draws where available..."
7,Country-level NCW,Primary wide country/account table with econom...,No has_gcw / has_bcw flags or standalone BCW m...
8,Review checks,Differences between the numbers in the supplie...,Review these rows before transferring numbers ...
9,Statistics,"Mean, sample SD (ddof=1), and median are compu...",These monetary draw summaries are not confiden...


## Save the complete manuscript workbook

This is intentionally the final cell. It writes every table above to a separate worksheet, applies filters, freeze panes, consistent headers and readable column widths, then confirms the output path and worksheet list.

In [13]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

workbook_tables = {
    "Read me": readme,
    "Table 1 Forest biomes": table1,
    "Table 2 Balance shifts": table2,
    "Table 3 Forest loss": table3,
    "Sources & methods": source_table,
    "Top 10 by metric": top_countries,
    "Unit crosswalk": unit_crosswalk,
    "Components": component_detail,
    "Country-level NCW": country_level_ncw,
    "Review checks": review_checks,
}

OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for sheet_index, (sheet_name, frame) in enumerate(workbook_tables.items(), start=1):
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.book[sheet_name]
        worksheet.freeze_panes = "A2"
        worksheet.sheet_view.showGridLines = False
        worksheet.sheet_properties.tabColor = "1B6B4B" if sheet_index <= 4 else "4472C4"
        if frame.shape[1] and frame.shape[0]:
            worksheet.auto_filter.ref = worksheet.dimensions
            table_name = "ManuscriptTable" + str(sheet_index)
            excel_table = Table(displayName=table_name, ref=worksheet.dimensions)
            excel_table.tableStyleInfo = TableStyleInfo(
                name="TableStyleMedium4",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False,
            )
            worksheet.add_table(excel_table)

        for cell in worksheet[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="17365D")
            cell.alignment = Alignment(vertical="center", wrap_text=True)
        worksheet.row_dimensions[1].height = 34

        for column_index, column_name in enumerate(frame.columns, start=1):
            values = [str(column_name)] + [str(value) for value in frame[column_name].head(250) if pd.notna(value)]
            width = min(max(max((len(value) for value in values), default=12) + 2, 12), 48)
            worksheet.column_dimensions[get_column_letter(column_index)].width = width
            if "URL" in str(column_name) or "Calculation" in str(column_name) or "Scope" in str(column_name):
                for row in worksheet.iter_rows(min_row=2, min_col=column_index, max_col=column_index):
                    row[0].alignment = Alignment(vertical="top", wrap_text=True)
                worksheet.column_dimensions[get_column_letter(column_index)].width = 48

        for row in worksheet.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)) and cell.value is not None:
                    cell.number_format = "#,##0.000;[Red]-#,##0.000;0"

check_workbook = pd.ExcelFile(OUTPUT_XLSX, engine="openpyxl")
if check_workbook.sheet_names != list(workbook_tables):
    raise RuntimeError(f"Workbook sheet check failed: {check_workbook.sheet_names}")
for sheet_name, frame in workbook_tables.items():
    saved = pd.read_excel(OUTPUT_XLSX, sheet_name=sheet_name, engine="openpyxl")
    if saved.shape != frame.shape:
        raise RuntimeError(f"Workbook dimensions changed for {sheet_name}: {saved.shape} != {frame.shape}")

print(f"Saved and checked: {OUTPUT_XLSX}")
print("Sheets:", " | ".join(check_workbook.sheet_names))
print(f"Workbook size: {OUTPUT_XLSX.stat().st_size:,} bytes")

Saved and checked: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo\results\tables\manuscript\Manuscript_tables.xlsx
Sheets: Read me | Table 1 Forest biomes | Table 2 Balance shifts | Table 3 Forest loss | Sources & methods | Top 10 by metric | Unit crosswalk | Components | Country-level NCW | Review checks
Workbook size: 498,506 bytes
